In [2]:
import pandas as pd
import numpy as np
import h5py

In [7]:
def h5py_prep(f, household_info):
    """
    Processes household load and heat pump data from HDF5 and merges with household metadata.

    Parameters:
    - f: HDF5 file object
    - household_info: DataFrame with household metadata

    Returns:
    - dfMerged: Processed load data with 'P_Load' and 'P_Pump' per household
    """

    def process_data(data_dict, system_type):
        df = pd.concat([
            pd.DataFrame(np.array(data_dict[key][system_type]["table"]))[["index", "P_TOT"]]
            .assign(building=key)
            for key in data_dict.keys()
        ], ignore_index=True)
        df['time'] = pd.to_datetime('1970-01-01') + pd.to_timedelta(df['index'], unit='s')
        df = df[["building", "time", "P_TOT"]]
        return df

    # Process household and heat pump data
    dfLoads = process_data(f["NO_PV"], "HOUSEHOLD")
    dfPumps = process_data(f["NO_PV"], "HEATPUMP")

    # Merge and rename columns
    dfLoads.rename(columns={"P_TOT": "P_Load"}, inplace=True)
    dfPumps.rename(columns={"P_TOT": "P_Pump"}, inplace=True)
    dfMerged = pd.merge(dfLoads, dfPumps, on=["building", "time"], how="outer")

    # Drop buildings with poor data quality
    droppedBuildings = ["SFH6", "SFH13", "SFH17", "SFH24", "SFH25", "SFH31", "SFH34", "SFH37", "SFH40"]
    dfMerged = dfMerged[~dfMerged['building'].isin(droppedBuildings)]

    # Standardize household ID format
    dfMerged.rename(columns={"building": "household_id"}, inplace=True)
    dfMerged['household_id'] = dfMerged['household_id'].str.extract('(\d+)').astype(int)

    # Convert power to kW and clip heat pump loads to physical maximum
    dfMerged['P_Load'] = dfMerged['P_Load'] / 1000
    dfMerged['P_Pump'] = dfMerged['P_Pump'] / 1000
    # dfMerged['P_Pump'] = dfMerged['P_Pump'].clip(upper=4.2)

    # Rename household_id to hh_id
    dfMerged = dfMerged.rename(columns={'household_id': 'hh_id',
                                        'P_Load': 'P_HH',
                                        'P_Pump': 'P_HP'})

    return dfMerged

<>:38: SyntaxWarning: invalid escape sequence '\d'
<>:38: SyntaxWarning: invalid escape sequence '\d'
C:\Users\55485\AppData\Local\Temp\ipykernel_24640\3477030160.py:38: SyntaxWarning: invalid escape sequence '\d'
  dfMerged['household_id'] = dfMerged['household_id'].str.extract('(\d+)').astype(int)


In [8]:
household_info = pd.read_csv("household_info.csv", sep=',')

In [9]:
f = h5py.File("2019_data_15min.hdf5", 'r')
load_data = h5py_prep(f, household_info)

In [10]:
load_data.to_csv("load_data.csv", sep=",", index=False)